# Milestone 2 — fine-tuning a separator on Sanidha

Trains the demixing network from `configs/default.yaml`, then charts what happened:
loss curves, per-stem validation SDR against the Milestone 1 HPSS baseline, and a
per-track evaluation of the selected checkpoint.

* **Run it top to bottom** to train. On a GPU, HTDemucs takes hours; every epoch is
  checkpointed, so an interrupted run resumes (set `RESUME = True`).
* **Set `RUN_TRAINING = False`** to only re-draw the charts from a finished run's
  `history.json` — no GPU needed.

Settings come from the config file; nothing below hard-codes a hyperparameter.

In [ ]:
import copy
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Run from the repository root whether Jupyter was started there or in notebooks/.
if not Path("configs").is_dir() and Path("../configs").is_dir():
    os.chdir("..")

from indian_mss.config import load_config

cfg = load_config("configs/default.yaml")


def deep_merge(base, extra):
    out = copy.deepcopy(base)
    for k, v in extra.items():
        out[k] = deep_merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
    return out


# Environment overrides exist for smoke-testing the notebook on synthetic data;
# for a real run, edit configs/default.yaml instead.
cfg = deep_merge(cfg, json.loads(os.environ.get("IMSS_CONFIG_OVERRIDES", "{}")))

SANIDHA_ROOT = os.environ.get("IMSS_SANIDHA_ROOT", cfg["data"]["sanidha_root"])
RUN_DIR = Path(os.environ.get("IMSS_RUN_DIR", f"checkpoints/{cfg['train']['backbone']}"))
RUN_TRAINING = os.environ.get("IMSS_RUN_TRAINING", "1") == "1"
RESUME = False
PRETRAINED = os.environ.get("IMSS_PRETRAINED", "1") == "1"

print(f"backbone {cfg['train']['backbone']}, data {SANIDHA_ROOT}, run dir {RUN_DIR}")

## 1. Dataset

In [ ]:
from indian_mss.data.sanidha import Sanidha
from indian_mss.train.report import dataset_stats

corpus = Sanidha(SANIDHA_ROOT, cfg["stft"]["sample_rate"], stems=tuple(cfg["data"]["stems"]))
stats = dataset_stats(corpus)
print(f"{stats['tracks']} tracks, {stats['hours']:.2f} h")
for stem, hours in stats["stem_hours"].items():
    print(f"  {stem:10s} {hours:6.2f} h")

## 2. Baseline to beat

The Milestone 1 HPSS numbers, from `python scripts/preprocess.py --evaluate`.
HPSS only splits melodic from percussive, so it is a floor, not a like-for-like
comparison with four instrument stems.

In [ ]:
from indian_mss.train.report import load_baseline

baseline = load_baseline("out/hpss_baseline.json")
if baseline is None:
    print("no out/hpss_baseline.json yet -- run: python scripts/preprocess.py --evaluate")
else:
    for name, m in baseline.items():
        print(f"HPSS {name:10s} SDR {m['sdr']:+.2f} dB  SIR {m['sir']:+.2f} dB  SAR {m['sar']:+.2f} dB")

## 3. Train

Phase 1 trains only the re-targeted output head with the pretrained network frozen;
phase 2 unfreezes the blocks listed in `train.unfreeze` at the 10× lower
`encoder_lr`. `best.pt` is chosen by mean validation SDR.

In [ ]:
if RUN_TRAINING:
    from indian_mss.train.loop import train

    history = train(cfg, SANIDHA_ROOT, RUN_DIR, pretrained=PRETRAINED, resume=RESUME, progress=True)
else:
    history = json.loads((RUN_DIR / "history.json").read_text())
print(f"{len(history['epochs'])} epochs, {len(history['steps'])} steps")

## 4. Loss curves

In [ ]:
from indian_mss.train.report import plot_losses

plot_losses(history)
plt.show()

## 5. Validation SDR per stem

Per stem, because a mean can hide a model that nails the vocal and fails the ghatam.

In [ ]:
from indian_mss.train.report import plot_val_sdr

plot_val_sdr(history, baseline)
plt.show()

## 6. Selected checkpoint on the held-out tracks

SDR, SIR and SAR per track and stem. SIR is the bleed number: how much of the
*other* instruments is left in each stem.

In [ ]:
from indian_mss.eval.sdr import evaluate_stems
from indian_mss.models.separator import load_separator, separate
from indian_mss.train.loop import _load_val_items

model, meta = load_separator(RUN_DIR / "best.pt")
print(f"best.pt: epoch {meta['epoch']}, backbone {meta['backbone']}")
items = _load_val_items(corpus, meta["val_tracks"], cfg["train"].get("val_seconds", 60.0), meta["sources"])

print(f"{'track':16s} {'stem':10s} {'SDR':>7s} {'SIR':>7s} {'SAR':>7s}")
for item in items:
    est = separate(model, item["mixture"], meta["stft_config"].sample_rate, meta["segment_seconds"])
    refs = {s: item["stems"][s] for s in meta["sources"] if s in item["stems"] and np.any(item["stems"][s])}
    scores = evaluate_stems({s: est[i] for i, s in enumerate(meta["sources"]) if s in refs}, refs)
    for stem, m in scores.items():
        print(f"{item['track_id']:16s} {stem:10s} {m['sdr']:+7.2f} {m['sir']:+7.2f} {m['sar']:+7.2f}")

## Next

* **Saraga** — the live-bleed generalization number needs `indian_mss/data/saraga.py`
  (not built yet). Expect a drop from the Sanidha figures above; the size of that drop
  is the result that matters.
* **Milestone 3** — `python scripts/separate.py --checkpoint <run>/best.pt --input mix.wav`
  writes the mask ratios on the Milestone 1 STFT grid, ready for resynthesis.